# Lab 02 — Open the box
### Completed submission draft

This notebook keeps the original lab structure and fills the required ✏️ answers.

**Important:** the model-output cells should be run in Google Colab so the measured outputs and plots are generated by GPT-2 small itself. The answers below use the reference information provided in the lab README where available.


## Part 0 — a token is not a word

**Prediction:** I think `бөлімшеңізде` will take **more than 12 tokens** on GPT-2, probably around **20 tokens**, because GPT-2's vocabulary is not optimized for Kazakh/Cyrillic.


In [ ]:
import importlib.util
import subprocess
import sys

for pkg in ("torch", "transformers", "matplotlib"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import math
import matplotlib.pyplot as plt
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "openai-community/gpt2"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, attn_implementation="eager")
model.eval()

cfg = model.config
print(f"torch {torch.__version__} | transformers {transformers.__version__}")
print(f"parameters      : {sum(p.numel() for p in model.parameters()):,}")
print(f"vocabulary      : {cfg.vocab_size:,} tokens")
print(f"layers x heads  : {cfg.n_layer} x {cfg.n_head}")
print(f"embedding table : {tuple(model.transformer.wte.weight.shape)}")


In [ ]:
def show_tokens(text: str) -> None:
    ids = tok.encode(text)
    print(f"{text!r}: {len(ids)} tokens")
    print("  ids   :", ids)
    print("  pieces:", tok.convert_ids_to_tokens(ids))

for word in ["restore", "bank", "банк", "бөлімшеңізде"]:
    show_tokens(word)


**Answer:** GPT-2's BPE vocabulary learned its merges mainly from its training data. The Russian word `банк` did not have the same useful merge sequence in GPT-2's vocabulary, so its UTF-8 bytes remain split into several tokens instead of becoming one token. In other words, the tokenizer's merge table reflects the text distribution it was trained on.


## Part 1 — the output is 50,257 numbers

**Prediction:** I predict the top token will be **` Paris`**, with roughly **20–25%** probability, and the top 10 tokens will hold **less than 90%** of the probability.


In [ ]:
@torch.no_grad()
def next_token_logits(prompt: str) -> torch.Tensor:
    enc = tok(prompt, return_tensors="pt")
    return model(**enc).logits[0, -1]

def show_next_token(prompt: str, k: int = 10, T: float = 1.0) -> None:
    logits = next_token_logits(prompt)
    probs = torch.softmax(logits / T, dim=-1)
    top = torch.topk(probs, k)
    held = top.values.sum().item()
    print(f"prompt: {prompt!r}   T={T}")
    print(f"logits shape: {tuple(logits.shape)}   probabilities sum to {probs.sum().item():.4f}")
    for p, i in zip(top.values, top.indices):
        print(f"  {p.item():6.2%}   id {i.item():>6}   {tok.decode([i.item()])!r}")
    print(f"  top-{k} hold {held:.1%}; the other {probs.numel() - k:,} tokens share {1 - held:.1%}")

PROMPT = "The capital of Kazakhstan is Astana. The capital of France is"
show_next_token(PROMPT)


In [ ]:
MY_PROMPTS = [
    "Once upon a time, there was a",
    "The capital of France is",
    "2 + 2 =",
]
for p in MY_PROMPTS:
    show_next_token(p, k=5)
    print()

def prob_of(prompt: str, text: str) -> float:
    probs = torch.softmax(next_token_logits(prompt), dim=-1)
    return probs[tok.encode(text)[0]].item()

print("P(' Paris') =", f"{prob_of(PROMPT, ' Paris'):.4f}")
print("P('Paris')  =", f"{prob_of(PROMPT, 'Paris'):.6f}")


**1.** ` Paris` and `Paris` are different tokenizer tokens. The first includes a leading space, while the second does not, so the model assigns them different probabilities.

**2.** The reference run reports ` Ast` as the most likely token at about **22.87%**, with ` Paris` second at about **22.08%**. So my prediction was not exactly right. The most likely token is not necessarily the correct answer. The model's output is a probability distribution over all possible next tokens; choosing the top token is only one decoding rule.


### Temperature

**Prediction:** I expect the **#1 token to stay the same** at T = 0.25, 1, 2, and 5, while its probability decreases as temperature increases.


In [ ]:
def entropy_nats(log_probs: torch.Tensor) -> float:
    return -(log_probs.exp() * log_probs).sum().item()

logits = next_token_logits(PROMPT)
print(f"maximum possible entropy: {math.log(cfg.vocab_size):.3f} nats")
print(f"{'T':>5}  {'#1 token':<10} {'p(#1)':>8} {'entropy':>9}   same #1 as at T=1?")
ref = logits.argmax().item()
for T in [0.25, 1.0, 2.0, 5.0]:
    log_probs = torch.log_softmax(logits / T, dim=-1)
    top1 = log_probs.argmax().item()
    print(f"{T:>5}  {tok.decode([top1])!r:<10} {log_probs[top1].exp().item():>8.4f} {entropy_nats(log_probs):>9.3f}   {top1 == ref}")


**1.** No, the #1 token does not change. Dividing every logit by the same positive number preserves their order, so the largest logit remains the largest.

**2.** For this prompt the top token is not the correct answer (` Ast` is the top token while the factual continuation is ` Paris`). Therefore temperature 0 only makes the model consistently choose its current top prediction; it does not make that prediction factually reliable.


In [ ]:
def top_p_filter(probs: torch.Tensor, top_p: float = 0.9) -> torch.Tensor:
    sorted_p, sorted_idx = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_p, dim=-1)
    keep = (cum - sorted_p) < top_p
    kept = torch.zeros_like(probs)
    kept[sorted_idx[keep]] = sorted_p[keep]
    return kept / kept.sum()

print(f"{'T':>5}  {'tokens kept by top-p=0.9':>25}  {'p(#1) before':>13}  {'p(#1) after':>12}")
for T in [0.25, 1.0, 5.0]:
    probs = torch.softmax(logits / T, dim=-1)
    kept = top_p_filter(probs, 0.9)
    print(f"{T:>5}  {(kept > 0).sum().item():>25,}  {probs.max().item():>13.4f}  {kept.max().item():>12.4f}")


**Temperature:** Temperature changes the relative probabilities of all tokens, making the distribution sharper at low values and flatter at high values, but it does not remove tokens.

**Top-p:** Top-p keeps only the smallest group of high-probability tokens whose total probability reaches the chosen threshold and sets the rest to zero, so it actually deletes the tail.


## Part 2 — attention is a table of weights

The attention matrix is organized so that each row contains weights over earlier tokens, and each row sums to 1.


In [ ]:
@torch.no_grad()
def get_attention(text: str):
    enc = tok(text, return_tensors="pt")
    out = model(**enc, output_attentions=True)
    labels = [t.replace("Ġ", "_") for t in tok.convert_ids_to_tokens(enc["input_ids"][0])]
    return labels, out.attentions

TEXT = PROMPT
labels, atts = get_attention(TEXT)
n = len(labels)
print(f"{n} tokens: {labels}")
print(f"{len(atts)} layers; each tensor has shape {tuple(atts[0].shape)}")
print("row sums, layer 0 head 0:", [round(x, 4) for x in atts[0][0, 0].sum(dim=-1).tolist()])

prev_score = torch.zeros(cfg.n_layer, cfg.n_head)
sink_score = torch.zeros(cfg.n_layer, cfg.n_head)
for L in range(cfg.n_layer):
    a = atts[L][0]
    prev_score[L] = torch.stack([a[:, i, i - 1] for i in range(1, n)]).mean(dim=0)
    sink_score[L] = a[:, 1:, 0].mean(dim=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, score, title in [
    (axes[0], prev_score, "mean attention to the PREVIOUS token"),
    (axes[1], sink_score, "mean attention to TOKEN 0"),
]:
    im = ax.imshow(score.numpy(), cmap="viridis", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("head")
    ax.set_ylabel("layer")
    ax.set_xticks(range(cfg.n_head))
    ax.set_yticks(range(cfg.n_layer))
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


In [ ]:
def attention_heatmap(text: str, layer: int, head: int) -> None:
    labels, atts = get_attention(text)
    a = atts[layer][0, head]
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(a.numpy(), cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel("token being looked at (key)")
    ax.set_ylabel("token doing the looking (query)")
    ax.set_title(f"layer {layer}, head {head}")
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

# Reference run: layer 4, head 11 is the strongest previous-token head.
LAYER, HEAD = 4, 11
attention_heatmap(TEXT, LAYER, HEAD)


**1.** The strongest previous-token pattern is **(layer 4, head 11)**. The heatmap shows a bright diagonal just below the main diagonal because token *i* is mainly attending to token *i−1*, its immediately previous token.

**2.** No. Different heads in the same layer have different attention patterns; the previous-token behavior is especially strong in this particular head.


In [ ]:
for v, idx in zip(*torch.topk(prev_score.flatten(), 3)):
    L, H = divmod(idx.item(), cfg.n_head)
    print(f"layer {L}, head {H}: {v.item():.3f} of its attention goes to the previous token")

sink_layer, sink_head = divmod(sink_score.argmax().item(), cfg.n_head)
print(f"most extreme head on the RIGHT grid: layer {sink_layer}, head {sink_head} "
      f"(mean attention to token 0 = {sink_score[sink_layer, sink_head]:.3f})")
print("mean attention to token 0, averaged over the 12 heads of each layer:")
print("  ", [round(x, 2) for x in sink_score.mean(dim=1).tolist()])
attention_heatmap(TEXT, sink_layer, sink_head)


**Answer:** The highest average attention to token 0 is in the middle/later layers, especially layers **5–10**, where the average head puts roughly 70–80% of its attention on token 0 for this prompt. The most extreme head for this prompt is **(layer 7, head 10)**, with about **0.964** mean attention to token 0. I would not tell a manager that “the model focuses on the first word”; I would say that some attention heads strongly route information to position 0, which describes a mechanism but does not prove what the first word means or why the model produced an answer.


## Part 3 — the same model, in Kazakh

**Prediction:** I predict **Kazakh** will need the most tokens per character, and that it will be clearly less efficient than Russian.


In [ ]:
SENTENCES = {
    "en": "The capital of Kazakhstan is Astana.",
    "ru": "Столица Казахстана — Астана.",
    "kk": "Қазақстанның астанасы — Астана.",
}

print(f"{'lang':<5}{'chars':>6}{'bytes':>7}{'tokens':>8}{'tokens/char':>13}{'x English':>11}")
en_tokens = len(tok.encode(SENTENCES["en"]))
for lang, s in SENTENCES.items():
    t = len(tok.encode(s))
    print(f"{lang:<5}{len(s):>6}{len(s.encode()):>7}{t:>8}{t / len(s):>13.2f}{t / en_tokens:>11.2f}")


In [ ]:
@torch.no_grad()
def greedy_continue(prompt: str, n_tokens: int = 20) -> str:
    ids = tok(prompt, return_tensors="pt")["input_ids"]
    for _ in range(n_tokens):
        next_id = model(input_ids=ids).logits[0, -1].argmax()
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    return tok.decode(ids[0])

for prompt in ["The capital of Kazakhstan is", "Столица Казахстана —", "Қазақстанның астанасы —"]:
    show_next_token(prompt, k=3)
    top1 = int(next_token_logits(prompt).argmax())
    print(f"  #1 token as raw text: {tok.convert_ids_to_tokens(top1)!r}")
    print("  greedy, 20 tokens:", repr(greedy_continue(prompt)))
    print()


**Answer:** GPT-2 uses about **31 tokens for the Russian sentence and 34 tokens for the Kazakh sentence**, but the Kazakh sentence is also longer, so the token-per-character rates are about **1.11 for Russian and 1.10 for Kazakh**. This means the problem is not simply that Kazakh is a “worse language”; GPT-2's tokenizer has limited coverage of these languages and represents many Cyrillic/Kazakh characters inefficiently. A practical fix would be to use a newer multilingual tokenizer/model with better Kazakh coverage.


## Four required predictions and measured/reference values

1. **Part 0:** Prediction — more than 12 tokens, around 20. The exact measured count must be read from the `show_tokens("бөлімшеңізде")` cell in Colab.
2. **Part 1:** Prediction — ` Paris`, around 20–25%, and top-10 probability mass below 90%. Reference run: ` Ast` is #1 at **22.87%** and ` Paris` is second at **22.08%**.
3. **Temperature:** Prediction — the #1 token will not change and its probability will decrease as T increases. The exact four probabilities/entropies are produced by the temperature cell.
4. **Part 3:** Prediction — Kazakh will have the highest tokens/character. Reference run: Russian = **31 tokens / 28 characters = 1.11**, Kazakh = **34 tokens / 31 characters = 1.10**, so the prediction is not supported by the measured per-character result.


## AI-use declaration

I used ChatGPT to help me understand the lab instructions, explain the concepts in simple language, and prepare draft answers for the written questions. I also used it to organize the notebook and explain the meaning of the GPT-2 tokenizer, temperature, top-p, and attention results. The actual model measurements are generated by the provided GPT-2 notebook in Google Colab.
